In [1]:
import re
import gurobipy as gp
from gurobipy import GRB

In [2]:
#NETWORK
network_file_path = "network.txt"

distance = {} #distance definition (from i to j)
nodes = set() #node set (all nodes)

#read network file
with open(network_file_path, "r") as network_file:
    #reading line by line
    for line in network_file:
        #removing extra spaces/line breaks
        line = line.strip()
        #skipping empty lines (safety check)
        if line == "":
            continue
        
        #splitting the line into three parts: i, j, d
        origin, destination, dist = line.split()
        
        #converting txt to numbres
        origin = int(origin)
        destination = int(destination)
        dist = float(dist)
        
        #noting distance between each OD pair
        distance[(origin, destination)] = dist
        
        #keep track of all nodes
        nodes.add(origin)
        nodes.add(destination)

print("Number of nodes found:", len(nodes))
print("Example distance entry:", list(distance.items())[:3])

Number of nodes found: 40
Example distance entry: [((0, 4), 291.0), ((4, 0), 291.0), ((0, 23), 99.0)]


In [3]:
#ROUTES
routes_file_path = "routes.txt"

routes_customers = []   # list of customers visited (WITHOUT depot)
route_distance  = []    # list of distance (cost) of each route

with open(routes_file_path, "r") as routes_file:
    for line in routes_file:
        line = line.strip()
        if line == "":
            continue
        
        numbers = [int(x) for x in re.findall(r"-?\d+", line)]
        
        #safety check: must have at least: distance, load, 0, 0
        if len(numbers) < 4:
            continue
        
        dist = numbers[0]          # first column: distance
        load = numbers[1]          # second column: load (not needed for 1a)
        path = numbers[2:]         # the route itself: starts/ends with depot 0
        
        #remove depot 0 if present at start/end
        if path[0] == 0:
            path = path[1:]
        if path and path[-1] == 0:
            path = path[:-1]
        
        routes_customers.append(tuple(path))
        route_distance.append(float(dist))

print("Number of routes read:", len(routes_customers))
print("First 3 routes (customers only):", routes_customers[:3])
print("First 3 route distances:", route_distance[:3])

Number of routes read: 14349
First 3 routes (customers only): [(14,), (1,), (6,)]
First 3 route distances: [932.0, 790.0, 1282.0]


In [4]:
#defining SETS
K = range(5) #number of vehicles given in assignment
R = range(len(routes_customers)) #number of feasible routes
customers = set()
for route in routes_customers:
    for i in route:
        customers.add(i)
customers = sorted(customers)

print("Nr of customers:", len(customers))
print("Customers:", customers)
print(range(len(routes_customers)))

Nr of customers: 25
Customers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
range(0, 14349)


In [5]:
#building the actual model
model = gp.Model("CVRP based cost minimization (1a)")

#decision bariable (4f) if vehicle k chooses route r {0,1} 
x = model.addVars(K, R, vtype=GRB.BINARY, name="x")

Set parameter Username
Set parameter LicenseID to value 2739960
Academic license - for non-commercial use only - expires 2026-11-17


In [6]:
#objective cost minimzation
    #with the assumption that route distance equals the cost!!!!!!!
model.setObjective(
    gp.quicksum(route_distance[r] *x[k,r] for k in K for r in R),
    GRB.MINIMIZE
)

In [7]:
#customer coverage constraint (4b) - each customer served once
for i in customers:
    model.addConstr(
        gp.quicksum(
            x[k,r] for k in K for r in R if i in routes_customers[r] ) ==1,
        name=f"cover_customer_{i}"
    )

In [8]:
#vehicle constraint (4d) - one route per vehicle
for k in K:
    model.addConstr(
        gp.quicksum(x[k,r] for r in R) == 1,
        name=f"one_route_per_verhicle_{k}"
    )

In [9]:
#solving the model
model.optimize()

#RESULTS
if model.status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
    chosen_routes = []

    for k in K:
        for r in R:
            if x[k, r].X > 0.5:
                chosen_routes.append(r)
                print(
                    f"Vehicle {k} → route {r}, "
                    f"customers {routes_customers[r]}, "
                    f"distance {route_distance[r]}"
                )

    costs = [route_distance[r] for r in chosen_routes]

    print("\nSummary:")
    print("Total cost:", sum(costs))
    print("Max route distance:", max(costs))
    print("Min route distance:", min(costs))
    print("Range (max − min):", max(costs) - min(costs))
    print("Runtime (s):", model.Runtime)
else:
    print("No fesible solution found.")


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i7-9750H CPU @ 2.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 30 rows, 71745 columns and 493020 nonzeros
Model fingerprint: 0x3b89ed47
Variable types: 0 continuous, 71745 integer (71745 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e+02, 3e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 0 rows and 42815 columns
Presolve time: 0.42s
Presolved: 30 rows, 28930 columns, 197780 nonzeros
Variable types: 0 continuous, 28930 integer (28930 binary)

Root relaxation: objective 8.703200e+03, 372 iterations, 0.16 seconds (0.37 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 8703.20000  

In [10]:
#PART 1B ----------------
epsilon = 0.05
z_star = 8831.0

K = range(5) #(repitiion)
R = range(len(routes_customers)) #feasible routes


In [29]:
#MAKING MODEL 1B
model_1b = gp.Model("Q1b_vehicle_index")

#using y instead of x so that the jupyter doesn't accidentally take the wrong one
y = model_1b.addVars(K, R, vtype=GRB.BINARY, name="y")

#add fairness variables
gamma = model_1b.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="gamma")
eta = model_1b.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="eta")

model_1b.setObjective(eta - gamma, GRB.MINIMIZE) #minimize range eta - gamma

In [30]:
#customer coverage constraint (4b)
for i in customers:
    model_1b.addConstr(
        gp.quicksum(y[k,r] for k in K for r in R if i in routes_customers[r]) ==1,
        name=f"cover_{i}"
    )

#vehicle route constraint (4d)
for k in K:
    model_1b.addConstr(
        gp.quicksum(y[k,r] for r in R) ==1,
        name=f"one_route_{k}"
        )

#efficiency constraint (cost should be lower than (1+e)z*) aka (105% requirement)
model_1b.addConstr(
    gp.quicksum(route_distance[r] * y[k,r] for k in K for r in R) <= ((1+epsilon)*z_star),
    name=f"cost_cap"
)

<gurobi.Constr *Awaiting Model Update*>

In [31]:
#DEFINING PAYOFFS
#payoff vehicle k is distance of its chosen route
for k in K:
    payoff_k =  gp.quicksum(route_distance[r] * y[k,r] for r in R)
    model_1b.addConstr(gamma <= payoff_k, name=f"gamma_def_{k}")
    model_1b.addConstr(payoff_k <= eta, name=f"eta_def_{k}")

In [32]:
#SOLVING THE MODEL
model_1b.optimize()

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i7-9750H CPU @ 2.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 41 rows, 71747 columns and 708265 nonzeros
Model fingerprint: 0xa7c56291
Variable types: 2 continuous, 71745 integer (71745 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 9e+03]
Presolve removed 0 rows and 5015 columns
Presolve time: 0.94s
Presolved: 41 rows, 66732 columns, 661135 nonzeros
Variable types: 0 continuous, 66732 integer (66730 binary)

Root relaxation: objective -2.273737e-13, 442 iterations, 0.42 seconds (0.88 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0    0.00000  

In [33]:
if model_1b.SolCount > 0:
    chosen = []
    costs = []
    
    print("\nChosen routes (vehicle-index):")
    for k in K:
        for r in R:
            if y[k, r].X > 0.5:
                chosen.append((k, r))
                costs.append(route_distance[r])
                print(f"Vehicle {k}: route {r} | customers {routes_customers[r]} | dist {route_distance[r]}")

    print("\nSummary (1b):")
    print("Total cost:", sum(costs))
    print("Cost cap:", (1.0 + epsilon) * z_star)
    print("Max route distance:", max(costs))
    print("Min route distance:", min(costs))
    print("Range (max-min):", max(costs) - min(costs))
    print("gamma:", gamma.X)
    print("eta:", eta.X)
    print("eta-gamma:", eta.X - gamma.X)
    print("Runtime (s):", model_1b.Runtime)
else:
    print("No feasible solution found within the time limit (this does NOT necessarily mean infeasible).")


Chosen routes (vehicle-index):
Vehicle 0: route 6841 | customers (13, 24, 3, 5, 2, 21) | dist 2463.0
Vehicle 1: route 604 | customers (8, 10, 25, 9) | dist 1000.0
Vehicle 2: route 10926 | customers (11, 22, 7, 14, 1) | dist 1529.0
Vehicle 3: route 4062 | customers (4, 17, 6, 23) | dist 1855.0
Vehicle 4: route 387 | customers (20, 18, 19, 12, 16, 15) | dist 2424.0

Summary (1b):
Total cost: 9271.0
Cost cap: 9272.550000000001
Max route distance: 2463.0
Min route distance: 1000.0
Range (max-min): 1463.0
gamma: 1000.0
eta: 2463.0
eta-gamma: 1463.0
Runtime (s): 380.6170001029968


In [39]:
#CORRECTNESS CHECK!!!!!!!!
#checking whether each vehicle picks one route 
for k in K:
    print("vehicle picks route", k, sum(y[k,r].X for r in R))

#is customer covered once?
for i in customers:
    cover = sum(y[k,r].X for k in K for r in R if i in routes_customers[r])
    if abs(cover - 1) > 1e-6:
        print("Customer coverage issue:", i, cover)
        break
else:
    print("All customers covered exactly once.")

#is cost cap respected?
total_cost = sum(route_distance[r] * y[k,r].X for k in K for r in R)
print("Total cost:", total_cost)
print("Cap:", 1.05 * z_star)

#does range match eta-gamma?
payoffs = []
for k in K:
    payoffs.append(sum(route_distance[r] * y[k,r].X for r in R))
print("Payoffs:", payoffs)
print("Range (max-min):", max(payoffs) - min(payoffs))
print("eta-gamma:", eta.X - gamma.X)


vehicle picks route 0 1.0
vehicle picks route 1 1.0
vehicle picks route 2 1.0
vehicle picks route 3 1.0
vehicle picks route 4 1.0
All customers covered exactly once.
Total cost: 9271.0
Cap: 9272.550000000001
Payoffs: [2463.0, 1000.0, 1529.0, 1855.0, 2424.0]
Range (max-min): 1463.0
eta-gamma: 1463.0
